# Dataset class counts by patient ID

This notebook reads `lung_us.yml`, iterates over the dataset images/labels,
and computes the number of YOLO class instances per patient ID.

Patient IDs are inferred from the beginning of each image filename,
using the same convention as `kfold_dataset_split.py` (prefix before the first underscore).

In [16]:
from pathlib import Path
from collections import defaultdict

import yaml
import pandas as pd

# Path to the dataset YAML used for training
data_yaml_path = Path('lung_us.yml').resolve()
data_yaml_path

WindowsPath('D:/Repos/aigt/UltrasoundObjectDetection/YOLOv8/lung_us.yml')

In [17]:
# Load YAML and resolve base dataset path and image directories
with data_yaml_path.open('r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

base_path = Path(data_cfg['path']).expanduser()
train = data_cfg.get('train')
val = data_cfg.get('val')
test = data_cfg.get('test')
names = data_cfg.get('names', {})

image_dirs = []
for sub in (train, val, test):
    if isinstance(sub, str) and sub:
        image_dirs.append(base_path / sub)

image_dirs

[WindowsPath('D:/Data/ObjectDetection/train/images'),
 WindowsPath('D:/Data/ObjectDetection/val/images')]

In [18]:
# Helper to infer patient ID from image filename
def patient_id_from_image_path(image_path: Path) -> str:
    """Extract patient ID from image filename.

    Assumes filenames start with the patient ID followed by an underscore,
    e.g. 'SCN22_frame001.png' -> patient ID 'SCN22'.
    """
    stem = image_path.stem
    return stem.split('_')[0]


IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}


def find_label_for_image(image_path: Path) -> Path:
    """Return the expected YOLO label path for a given image.

    Assumes this fixed layout:
    - Images are under a directory named 'images'
    - Labels are under a sibling directory named 'labels'
    - Label filename is identical to the image stem with '.txt' extension

    Example:
    D:/Data/ObjectDetection/all/train/images/SCN1_frame001.png
    -> D:/Data/ObjectDetection/all/train/labels/SCN1_frame001.txt
    """
    parts = list(image_path.parts)

    # Find the last occurrence of 'images' and replace it with 'labels'
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == 'images':
            parts[i] = 'labels'
            labels_dir = Path(*parts).parent  # directory .../labels
            return labels_dir / (image_path.stem + '.txt')

    # If 'images' is not present for some reason, fall back to same directory
    return image_path.with_suffix('.txt')


find_label_for_image(Path('D:/Data/ObjectDetection/all/train/images/SCN1_frame001.png'))

WindowsPath('D:/Data/ObjectDetection/all/train/labels/SCN1_frame001.txt')

In [19]:
# Iterate over all images and accumulate class counts per patient ID
patient_class_counts = defaultdict(lambda: defaultdict(int))
total_missing_label_files = 0
total_images = 0

for img_dir in image_dirs:
    if not img_dir.exists():
        continue
    for p in img_dir.rglob('*'):
        if not p.is_file():
            continue
        if p.suffix.lower() not in IMG_EXTENSIONS:
            continue

        total_images += 1
        pid = patient_id_from_image_path(p)
        label_path = find_label_for_image(p)

        if not label_path.exists():
            total_missing_label_files += 1
            continue

        with label_path.open('r', encoding='utf-8') as lf:
            for line in lf:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                try:
                    cls = int(parts[0])
                except (ValueError, IndexError):
                    continue
                patient_class_counts[pid][cls] += 1

total_images, total_missing_label_files


(1097, 6)

In [20]:
# Convert to a DataFrame: rows = patients, columns = class IDs (and totals)
all_patient_ids = sorted(patient_class_counts.keys())
all_class_ids = sorted({c for counts in patient_class_counts.values() for c in counts.keys()})

rows = []
for pid in all_patient_ids:
    row = {'patient_id': pid}
    total_for_patient = 0
    for cid in all_class_ids:
        count = patient_class_counts[pid].get(cid, 0)
        row[f'class_{cid}'] = count
        total_for_patient += count
    row['total_instances'] = total_for_patient
    rows.append(row)

# Build DataFrame; handle the case where there are no rows
if rows:
    df = pd.DataFrame(rows)
    df = df.sort_values(by='total_instances', ascending=False).reset_index(drop=True)
else:
    # Empty DataFrame with the expected columns
    df = pd.DataFrame(columns=['patient_id', 'total_instances'])

df

,patient_id,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,total_instances
0,SCN2,455,789,124,355,15,0,0,0,1738
1,SCN1,313,529,48,135,380,257,0,29,1691
2,SCN17,52,724,4,233,50,211,72,14,1360
3,SCN22,383,0,0,0,336,28,0,0,747


In [21]:
# Optional: map class column names to human-readable labels from the YAML
rename_map = {}
for col in df.columns:
    if col.startswith('class_') and col[len('class_'):].isdigit():
        cid = int(col[len('class_'):])
        class_name = names.get(cid, None)
        if class_name is not None:
            rename_map[col] = f'{col}__{class_name}'

df_renamed = df.rename(columns=rename_map)
df_renamed


,patient_id,class_0__Pleura-intact,class_1__Pleura-affected,class_2__B-lines,class_3__B-lines-septal or glass rockets,class_4__A-lines,class_5__Consolidation-nontranslobar,class_6__Consolidation-translobar,class_7__Pleural effusion,total_instances
0,SCN2,455,789,124,355,15,0,0,0,1738
1,SCN1,313,529,48,135,380,257,0,29,1691
2,SCN17,52,724,4,233,50,211,72,14,1360
3,SCN22,383,0,0,0,336,28,0,0,747


In [22]:
def compute_class_counts_by_split() -> pd.DataFrame:
    """Return class instance counts per dataset split.

    Rows: human-readable class names from the YAML `names` mapping.
    Columns: 'Training Data', 'Validation Data', 'Total'.
    """
    # Map each configured split to a name and its corresponding images directory
    split_dirs = []
    for split_key, split_name in (("train", "Training Data"), ("val", "Validation Data")):
        sub = data_cfg.get(split_key)
        if isinstance(sub, str) and sub:
            img_dir = base_path / sub
            split_dirs.append((split_name, img_dir))

    # Initialize counts dict: {human_class_name: {split_name: count}}
    class_split_counts = defaultdict(lambda: defaultdict(int))

    # Ensure we know the mapping from class id to human-readable name
    # `names` may be a list or dict, handle both via dict-like access
    def class_id_to_name(cid: int) -> str:
        label = names.get(cid, str(cid)) if isinstance(names, dict) else (
            names[cid] if 0 <= cid < len(names) else str(cid)
        )
        return str(label)

    for split_name, img_dir in split_dirs:
        if not img_dir.exists():
            continue
        for p in img_dir.rglob('*'):
            if not p.is_file():
                continue
            if p.suffix.lower() not in IMG_EXTENSIONS:
                continue

            label_path = find_label_for_image(p)
            if not label_path.exists():
                continue

            with label_path.open('r', encoding='utf-8') as lf:
                for line in lf:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split()
                    try:
                        cls = int(parts[0])
                    except (ValueError, IndexError):
                        continue

                    class_name = class_id_to_name(cls)
                    class_split_counts[class_name][split_name] += 1

    # Build DataFrame
    all_class_names = sorted(class_split_counts.keys())
    rows = []
    for cname in all_class_names:
        train_count = class_split_counts[cname].get("Training Data", 0)
        val_count = class_split_counts[cname].get("Validation Data", 0)
        total = train_count + val_count
        rows.append({
            "class_name": cname,
            "Training Data": train_count,
            "Validation Data": val_count,
            "Total": total,
        })

    if rows:
        df_split = pd.DataFrame(rows).set_index("class_name")
    else:
        df_split = pd.DataFrame(columns=["Training Data", "Validation Data", "Total"])

    return df_split

# Example usage:
compute_class_counts_by_split()

,Training Data,Validation Data,Total
class_name,,,
A-lines,401,380,781
B-lines,128,48,176
B-lines-septal or glass rockets,588,135,723
Consolidation-nontranslobar,239,257,496
Consolidation-translobar,72,0,72
Pleura-affected,1513,529,2042
Pleura-intact,890,313,1203
Pleural effusion,14,29,43


In [24]:
def class_counts_df_to_latex(
    df_split: pd.DataFrame,
    caption: str = "A teljes adathalmaz lebontása tanítási/tesztelési, valamint klasszifikációs osztály szerint",
    label: str = "tab:adat_felosztas",
    arraystretch: float = 1.5,
) -> str:
    """Convert class count DataFrame into a LaTeX table.

    Expects a DataFrame like the output of `compute_class_counts_by_split()`:
    - Index: class names (human readable)
    - Columns: 'Training Data', 'Validation Data', 'Total'

    Returns a LaTeX table string with Hungarian headers, e.g.:
    - Tanító adathalmaz (Training Data)
    - Teszt adathalmaz (Validation Data)
    - Összesen (Total)
    """

    # Ensure the index contains the class names
    if "class_name" in df_split.columns and df_split.index.name is None:
        df_local = df_split.set_index("class_name")
    else:
        df_local = df_split.copy()

    # Column sums for the final "Összesen" row
    train_sum = int(df_local["Training Data"].sum()) if "Training Data" in df_local.columns else 0
    val_sum = int(df_local["Validation Data"].sum()) if "Validation Data" in df_local.columns else 0
    total_sum = int(df_local["Total"].sum()) if "Total" in df_local.columns else train_sum + val_sum

    lines = []
    lines.append(f"\\def\\arraystretch{{{arraystretch}}}")
    lines.append("\\begin{table}[htb]")
    lines.append("    \\centering")
    lines.append("    \\begin{tabular}{lccc}")
    lines.append("        \\hline")
    lines.append("        & Tanító adathalmaz & Teszt adathalmaz & Összesen \\\\ \\hline \\hline")

    # Data rows: one per class
    for cname, row in df_local.iterrows():
        train_val = int(row.get("Training Data", 0))
        val_val = int(row.get("Validation Data", 0))
        total_val = int(row.get("Total", train_val + val_val))
        lines.append(f"        {cname} & {train_val} & {val_val} & {total_val}\\\\ \\hline")

    # Final summary row
    lines.append(f"        Összesen & {train_sum} & {val_sum} & {total_sum}\\\\")
    lines.append("        \\hline")
    lines.append("    \\end{tabular}")
    lines.append(f"    \\caption{{{caption}}}")
    lines.append(f"    \\label{{{label}}}")
    lines.append("\\end{table}")

    return "\n".join(lines)

# Example usage (after computing df_split):
df_split = compute_class_counts_by_split()
latex_table = class_counts_df_to_latex(df_split)
print(latex_table)

\def\arraystretch{1.5}
\begin{table}[htb]
    \centering
    \begin{tabular}{lccc}
        \hline
        & Tanító adathalmaz & Teszt adathalmaz & Összesen \\ \hline \hline
        A-lines & 401 & 380 & 781\\ \hline
        B-lines & 128 & 48 & 176\\ \hline
        B-lines-septal or glass rockets & 588 & 135 & 723\\ \hline
        Consolidation-nontranslobar & 239 & 257 & 496\\ \hline
        Consolidation-translobar & 72 & 0 & 72\\ \hline
        Pleura-affected & 1513 & 529 & 2042\\ \hline
        Pleura-intact & 890 & 313 & 1203\\ \hline
        Pleural effusion & 14 & 29 & 43\\ \hline
        Összesen & 3845 & 1691 & 5536\\
        \hline
    \end{tabular}
    \caption{A teljes adathalmaz lebontása tanítási/tesztelési, valamint klasszifikációs osztály szerint}
    \label{tab:adat_felosztas}
\end{table}
